##**Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend**

###**Task 4.1 — One validated call**


In [1]:
# ------------------------------------------------------------
# 1. Install required libraries
# ------------------------------------------------------------
!pip install -q google-genai pydantic

In [ ]:
# ------------------------------------------------------------
# 2. Import libraries
# ------------------------------------------------------------
import os
import json
import pandas as pd
from google import genai
from google.genai import types
# ------------------------------------------------------------
# 3. Define the JSON schema
# ------------------------------------------------------------
SCHEMA = {
    "type": "object",

    "properties": {
        "root_cause": {
            "type": "string"
        },

        "confidence": {
            "type": "number"
        },

        "summary": {
            "type": "string"
        },

        "recommended_action": {
            "type": "string"
        }
    },

    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}
# ------------------------------------------------------------
# 4. Gemini API Setup
# ------------------------------------------------------------

API_KEY = input("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)
# ------------------------------------------------------------
# 5. Define the system instruction
# ------------------------------------------------------------

SYSTEM_TEXT = """
You are an AI assistant working for CareLoop Clinics.

Your job is to analyse patient follow-up notes
and identify why patients missed their appointments.

Return accurate and structured information.
Do not invent information that is not present in the note.
"""

# ------------------------------------------------------------
# 6. Create the analyse_note() function
# ------------------------------------------------------------

def analyse_note(note_text, return_raw=False):

    prompt = f"""
    Context:
    CareLoop Clinics wants to understand why patients
    miss their medical appointments.

    Role:
    You are a patient follow-up note analyst.

    Action:
    Analyse the following patient note and identify
    the main reason for missing the appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    If the reason is unclear, select other.
    Do not guess.

    Tone:
    Professional, clear and factual.

    Format:
    Return JSON containing exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    The summary must contain at most 25 words.

    Patient note:
    {note_text}
    """
    MODEL_NAME = "gemini-3.5-flash-lite"
    response = client.models.generate_content(

        model= MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=SYSTEM_TEXT,

            temperature=0,

            max_output_tokens=1000,

            response_mime_type="application/json",

            response_schema=SCHEMA,

            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)

    if return_raw:
      return data, response.text

    return data

# ------------------------------------------------------------
# 7. Read N001 from followup_notes.csv
# ------------------------------------------------------------
df = pd.read_csv("followup_notes.csv")

note = df.loc[
    df["note_id"] == "N001",
    "note_text"
].iloc[0]

print("Patient Note:", note)
# ------------------------------------------------------------
# 8. Call the function
# ------------------------------------------------------------

result = analyse_note(note)
print(result)


#**Task 4.2 — Validate and process the 10 notes**


In [5]:
# ------------------------------------------------------------
# 1. Create the Pydantic model
# ------------------------------------------------------------
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)
# ------------------------------------------------------------
# 2. Define the allowed root causes
# ------------------------------------------------------------
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]
# ------------------------------------------------------------
# 3. Process the first 10 notes
# ------------------------------------------------------------
import pandas as pd
import numpy as np
import time
import json

# Read the CSV file
df = pd.read_csv("followup_notes.csv")

# Take only the first 10 rows
df = df.head(10)

results = []
api_calls = 0

for index, row in df.iterrows():
    note_text = row["note_text"]

    # Create a default result
    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": np.nan,
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check whether note is blank
    if pd.isna(note_text) or str(note_text).strip() == "":
        result["root_cause"] = "other"
        result["confidence"] = 0.0
        result["summary"] = "No note recorded"
        result["recommended_action"] = "Re-call patient"
        result["validation_status"] = "skipped_blank"
    else:
        try:
            # Call Gemini and get dictionary + original response
            data, raw_response = analyse_note(
                note_text,
                return_raw=True
            )
            api_calls += 1
            result["raw_response"] = raw_response

            # Copy fields actually returned by Gemini
            for field in ["root_cause", "confidence", "summary", "recommended_action"]:
                result[field] = data.get(field, np.nan if field == "confidence" else "")

            # Pydantic validation
            validated_data = FollowupInsight(**data)

            # Check allowed root cause
            if validated_data.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError("Invalid root_cause value")

            result["validation_status"] = "valid"
        except Exception as e:
            result["validation_status"] = "invalid"
            print("Invalid response:", e)

        # Pause between API calls to respect free-tier limits
        if api_calls < 9:
            time.sleep(13)

    results.append(result)

# Convert results into DataFrame and save
results_df = pd.DataFrame(results)

# Safe type coercion so empty fields do not break the background styler
results_df["confidence"] = pd.to_numeric(results_df["confidence"], errors='coerce')
results_df.to_csv("notes_analysed.csv", index=False)

# Beautifully display the Task 4.2 table safely
print(f"--- Analyzed Patient Notes (Total API Calls: {api_calls}) ---")
styled_results = results_df[["note_id", "appointment_id", "root_cause", "confidence", "summary", "recommended_action", "validation_status"]].style \
    .background_gradient(cmap="Blues", subset=["confidence"]) \
    .format({"confidence": "{:.2f}"}, na_rep="N/A") \
    .set_properties(**{'text-align': 'left'}) \
    .set_caption("First 10 Analyzed & Validated Patient Follow-up Notes")
display(styled_results)

--- Analyzed Patient Notes (Total API Calls: 9) ---


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status
0,N001,A0872,forgot_or_no_reminder,1.00,Patient missed the appointment because she did not receive a reminder and forgot.,Reschedule the appointment for next week and verify contact information.,valid
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they live far from the clinic and could not find transportation.,Offer telehealth options or assist with transport arrangements for future visits.,valid
2,N003,A0526,cost_or_fee,0.99,Patient skipped the appointment because the consultation fee was too high and inquired about available discounts.,"Provide information regarding financial assistance, payment plans, or available discounts for follow-up visits.",valid
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank
4,N005,A0534,illness_or_emergency,0.95,"Patient missed the appointment due to a high fever and body ache, and remained in bed.",Reschedule the appointment once the patient has fully recovered.,valid
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgent work meeting and requested an evening slot.,Reschedule the appointment for an evening slot.,valid
6,N007,A0333,forgot_or_no_reminder,0.95,The patient forgot the appointment because they did not receive an SMS reminder and requested a rebooking.,Reschedule the appointment and verify the patient's mobile number for SMS notifications.,valid
7,N008,A0222,distance_or_transport,0.95,"The patient missed the appointment due to a city bus strike and a lack of available autos, compounded by distance.",Offer telehealth options or assistance with transportation planning for future visits.,valid
8,N009,A0977,other,0.50,Patient could not be reached after three attempts because their phone was switched off.,Try contacting the patient via an alternative communication method such as email or mail.,valid
9,N010,A0596,work_or_schedule,0.95,The patient missed the appointment due to a factory shift change and inability to take leave.,Reschedule future appointments for Saturdays as requested by the patient.,valid


#**Task 4.3 — Break it on purpose**


In [6]:
# ------------------------------------------------------------
# Part A — Test Pydantic with bad responses
# ------------------------------------------------------------
from pydantic import ValidationError

# Test 1: Invalid confidence
bad_data_1 = {
    "root_cause": "cost_or_fee",
    "confidence": 1.7,
    "summary": "Patient could not afford the fee.",
    "recommended_action": "Offer payment options."
}

try:
    result = FollowupInsight(**bad_data_1)
    print(result)

except ValidationError as e:
    print("Validation Error 1:")
    print(e)


# Test 2: Missing recommended_action
bad_data_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.9,
    "summary": "Patient could not afford the fee."
}

try:
    result = FollowupInsight(**bad_data_2)
    print(result)

except ValidationError as e:
    print("Validation Error 2:")
    print(e)


Validation Error 1:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Validation Error 2:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ld not afford the fee.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [7]:
# ------------------------------------------------------------
# Part B — Compare two prompts
# ------------------------------------------------------------

# 1: Create a function for the prompt comparison
def classify_with_prompt(note_text, prompt):
    MODEL_NAME = "gemini-3.5-flash-lite"
    response = client.models.generate_content(
        model= MODEL_NAME,
        contents=prompt + "\nPatient note: " + note_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    data = json.loads(response.text)
    return data

# 2: Prepare N007 and N009
df = pd.read_csv("followup_notes.csv")
note_007 = df.loc[df["note_id"] == "N007", "note_text"].iloc[0]
note_009 = df.loc[df["note_id"] == "N009", "note_text"].iloc[0]

# 3: Run all four comparisons
V1 = "Classify this note:"
V2 = """
You are a healthcare follow-up note classification assistant.
Classify the patient's reason for missing the appointment.
Allowed root causes:
forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
If the note is ambiguous or the reason is unclear, choose other rather than guessing.
Return only JSON using the required schema.
"""

comparison_results = []
tests = [
    ("N007", note_007, "V1", V1),
    ("N007", note_007, "V2", V2),
    ("N009", note_009, "V1", V1),
    ("N009", note_009, "V2", V2)
]

for note_id, note_text, version, prompt in tests:
    try:
        data = classify_with_prompt(note_text, prompt)
        root_cause = data.get("root_cause", "")
        status = "Allowed" if root_cause in ALLOWED_ROOT_CAUSES else "Not Allowed"
        comparison_results.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": root_cause,
            "status": status
        })
    except Exception as e:
        comparison_results.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": "",
            "status": f"Error: {e}"
        })
    time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)

# Beautifully display comparison results
print("--- Prompt Comparison Results (V1 vs V2) ---")
styled_comparison = comparison_df.style \
    .set_properties(**{'text-align': 'center'}) \
    .set_caption("Comparison of Default vs CRAFT Prompt Performance")
display(styled_comparison)

print("\n--- Recommendation ---")
print("Deploy Prompt Version: V2")
print("Why: V2 systematically guides the model using the CRAFT framework, strictly enforces allowed classes, prevents guessing on ambiguous records (like N009), and correctly structures critical healthcare insights.")

# ------------------------------------------------------------
# Part C — Count total API calls
# ------------------------------------------------------------
task_4_1_calls = 1
task_4_2_calls = api_calls
task_4_3_calls = 4
total_api_calls = task_4_1_calls + task_4_2_calls + task_4_3_calls

print(f"\nAPI usage counters:\n - Task 4.1 calls: {task_4_1_calls}\n - Task 4.2 calls: {task_4_2_calls}\n - Task 4.3 calls: {task_4_3_calls}\n - Total API calls: {total_api_calls}")

--- Prompt Comparison Results (V1 vs V2) ---


,note_id,prompt,root_cause,status
0,N007,V1,forgot_appointment_no_reminder,Not Allowed
1,N007,V2,forgot_or_no_reminder,Allowed
2,N009,V1,unreachable,Not Allowed
3,N009,V2,other,Allowed



--- Recommendation ---
Deploy Prompt Version: V2
Why: V2 systematically guides the model using the CRAFT framework, strictly enforces allowed classes, prevents guessing on ambiguous records (like N009), and correctly structures critical healthcare insights.

API usage counters:
 - Task 4.1 calls: 1
 - Task 4.2 calls: 9
 - Task 4.3 calls: 4
 - Total API calls: 14


#**Task 4.4 — Recommendation and GitHub**


In [8]:
# ------------------------------------------------------------
# Part A — Merge the two CSV files
# ------------------------------------------------------------

# 1: Read both CSV files
import pandas as pd

notes = pd.read_csv("notes_analysed.csv")
appointments = pd.read_csv("appointments_clean.csv")

print(f"Notes shape: {notes.shape}")
print(f"Appointments shape: {appointments.shape}\n")

# 2: Merge the files
merged_df = pd.merge(
    notes,
    appointments[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("--- Merged DataFrame Preview ---")
display(merged_df[["note_id", "appointment_id", "root_cause", "reminder_sent", "confidence", "summary"]].style.set_properties(**{'text-align': 'left'}))

# ------------------------------------------------------------
# Part B — Cross-tabulate root cause against reminder status
# ------------------------------------------------------------

cross_tab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    margins=True
)

print("\n--- Structured Cross-Tabulation ---")
display(cross_tab.style.background_gradient(cmap='Blues').set_caption("Root Cause vs. Reminder Status Cross-tabulation"))

Notes shape: (10, 8)
Appointments shape: (990, 16)

--- Merged DataFrame Preview ---


,note_id,appointment_id,root_cause,reminder_sent,confidence,summary
0,N001,A0872,forgot_or_no_reminder,No,1.000000,Patient missed the appointment because she did not receive a reminder and forgot.
1,N002,A0897,distance_or_transport,No,0.950000,Patient missed the appointment because they live far from the clinic and could not find transportation.
2,N003,A0526,cost_or_fee,Yes,0.990000,Patient skipped the appointment because the consultation fee was too high and inquired about available discounts.
3,N004,A0926,other,No,0.000000,No note recorded
4,N005,A0534,illness_or_emergency,Yes,0.950000,"Patient missed the appointment due to a high fever and body ache, and remained in bed."
5,N006,A0507,work_or_schedule,Yes,0.950000,Patient missed the appointment due to an urgent work meeting and requested an evening slot.
6,N007,A0333,forgot_or_no_reminder,No,0.950000,The patient forgot the appointment because they did not receive an SMS reminder and requested a rebooking.
7,N008,A0222,distance_or_transport,Yes,0.950000,"The patient missed the appointment due to a city bus strike and a lack of available autos, compounded by distance."
8,N009,A0977,other,Yes,0.500000,Patient could not be reached after three attempts because their phone was switched off.
9,N010,A0596,work_or_schedule,No,0.950000,The patient missed the appointment due to a factory shift change and inability to take leave.



--- Structured Cross-Tabulation ---


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10
